- Deadline of submission: 23h59, 01/06/2026
- Submission [here](https://nextcloud.centralesupelec.fr/s/LyG47JMLMJKqZGf)
- Please name your submission as `td8_firstname_lastname.ipynb`

# Exercise 1: Train a regression model to predict the motor temperature in the data challenge

In this exercise, you will use `run_cv_one_motor` with `mdl_type='reg'` to train a regression model to predict the motor temperature. The goal is to understand how to make more accurate prediction thtough a rolling horizon-based approach.

As the previous TD, we work on the training dataset. Since we work on regression model, we use all the datasets that have no failures. Below is a code snippet to load the training dataset. 

In [1]:
utility_path = './kaggle_data_challenge/'
import sys
sys.path.insert(1, utility_path)

from utility import read_all_test_data_from_path
import numpy as np
import pandas as pd


def pre_processing(df: pd.DataFrame):
    ''' # Description
    Remove outliers from the dataframe based on defined valid ranges. 
    Define a valid range of temperature and voltage. 
    Use ffil function to replace the invalid measurement with the previous value.
    '''
    df['temperature'] = df['temperature'].where(df['temperature'] <= 100, np.nan)
    df['temperature'] = df['temperature'].where(df['temperature'] >= 0, np.nan)
    df['temperature'] = df['temperature'].ffill()

    df['voltage'] = df['voltage'].where(df['voltage'] >= 6000, np.nan)
    df['voltage'] = df['voltage'].where(df['voltage'] <= 9000, np.nan)
    df['voltage'] = df['voltage'].ffill()

    df['position'] = df['position'].where(df['position'] >= 0, np.nan)
    df['position'] = df['position'].where(df['position'] <= 1000, np.nan)
    df['position'] = df['position'].ffill()


base_dictionary = './kaggle_data_challenge/kaggle_data_challenge/training_data/'
df_data = read_all_test_data_from_path(base_dictionary, pre_processing, is_plot=False)

# Get all the normal samples.
normal_test_id = ['20240105_164214', 
    '20240105_165300', 
    '20240105_165972', 
    '20240320_152031', 
    '20240320_153841', 
    '20240320_155664', 
    '20240321_122650', 
    '20240325_135213', 
    '20240426_141190', 
    '20240426_141532', 
    '20240426_141602', 
    '20240426_141726', 
    '20240426_141938', 
    '20240426_141980', 
    '20240503_164435']
df_data = df_data[df_data['test_condition'].isin(normal_test_id)]

### Use `run_cv_one_motor` to run cross validation.

Like in the classification-based approach, we use `run_cv_one_motor` to run cross validation. The only difference is that we need to set `mdl_type = 'reg'`. 

You can try to change the parameters window_size, sample_step and prediction_lead_time as well:
- window_size: the size of the sliding window. The features in the sliding window are concanetated into a feature vector.
- sample_step: we allow taking features evvery `sample_step` data points.
- prediction_lead_time: the lead time of the prediction. The model is trained to predict the temperature `prediction_lead_time` data points later.

You can set the parameter value `threshold` to control the threshold of residual error. If the absolute value of the residual error is larger than the threshold, the corresponding data point can be marked as failure. If you do not specify, `threshold` will take its default value of $3.$

If you set `single_run_result = False`, only the final results will be printed.

Below is a demo on motor 6:

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from utility import run_cv_one_motor


# Define the steps of the pipeline
steps = [
    ('standardizer', StandardScaler()),  # Step 1: StandardScaler
    ('regressor', LinearRegression())    # Step 2: Linear Regression
]

# Create the pipeline
mdl_linear_regreession = Pipeline(steps)

feature_list_all = ['time', 'data_motor_1_position', 'data_motor_1_temperature', 'data_motor_1_voltage',
                    'data_motor_2_position', 'data_motor_2_temperature', 'data_motor_2_voltage',
                    'data_motor_3_position', 'data_motor_3_temperature', 'data_motor_3_voltage',
                    'data_motor_4_position', 'data_motor_4_temperature', 'data_motor_4_voltage',
                    'data_motor_5_position', 'data_motor_5_temperature', 'data_motor_5_voltage',
                    'data_motor_6_position', 'data_motor_6_temperature', 'data_motor_6_voltage']

threshold = 1
window_size = 1
sample_step = 1
prediction_lead_time = 1

df_perf = run_cv_one_motor(motor_idx=6, df_data=df_data, mdl=mdl_linear_regreession, 
            feature_list=feature_list_all, n_fold=5, 
            threshold=threshold, window_size=window_size, sample_step=sample_step, prediction_lead_time=prediction_lead_time,
            single_run_result=False, mdl_type='reg')

Model for motor 6:


   Max error       RMSE  Exceed boundary rate
0  14.209864  42.292243              0.818775
1  14.916205  30.510211              0.995528
2   8.129535  17.869962              0.632314
3  11.186461  26.657576              0.838308
4   9.631486  26.782041              0.877353


Mean performance metric and standard error:
Max error: 11.6147 +- 2.9111
RMSE: 28.8224 +- 8.8472
Exceed boundary rate: 0.8325 +- 0.1312




1. Explain what does "Exceed boundary rate" mean? Do you think we can use such a regression model to support fault detection? Why?

The exceed boundary rate is the proportion of points whose absolute residual error (|prediction - measurement|) is larger than the threshold. It measures how often the prediction deviates from the real value by more than the margin we allow.

This type of model can be used for fault detection. We train it only on normal data, so it learns the temperature behavior of the motor under healthy conditions. When a fault appears, the measured temperature departs from the predicted value and the residual grows beyond the threshold. A high exceed boundary rate on new data therefore indicates abnormal behavior, and the threshold is the criterion we use to classify each point as normal or faulty.

2. Try a larger window size `window_size`, and see if the performance is better. Explain why.

In [3]:
window_sizes_to_test = [1, 5, 10, 20]
results = {}

for ws in window_sizes_to_test:
    mdl_tmp = Pipeline([
        ('standardizer', StandardScaler()),
        ('regressor', LinearRegression())
    ])

    df_perf_tmp = run_cv_one_motor(motor_idx=6, df_data=df_data, mdl=mdl_tmp,
                feature_list=feature_list_all, n_fold=5,
                threshold=threshold, window_size=ws, sample_step=1, prediction_lead_time=1,
                single_run_result=False, mdl_type='reg')
    results[ws] = df_perf_tmp

print("\n=== Summary ===")
for ws, perf in results.items():
    print(f"\nwindow_size = {ws}:")
    print(f"  Max error:           {perf['Max error'].mean():.4f} +- {perf['Max error'].std():.4f}")
    print(f"  RMSE:                {perf['RMSE'].mean():.4f} +- {perf['RMSE'].std():.4f}")
    print(f"  Exceed boundary rate: {perf['Exceed boundary rate'].mean():.4f} +- {perf['Exceed boundary rate'].std():.4f}")

Model for motor 6:


   Max error       RMSE  Exceed boundary rate
0  14.209864  42.292243              0.818775
1  14.916205  30.510211              0.995528
2   8.129535  17.869962              0.632314
3  11.186461  26.657576              0.838308
4   9.631486  26.782041              0.877353


Mean performance metric and standard error:
Max error: 11.6147 +- 2.9111
RMSE: 28.8224 +- 8.8472
Exceed boundary rate: 0.8325 +- 0.1312


Model for motor 6:


   Max error      RMSE  Exceed boundary rate
0   1.015867  0.026922              0.003262
1   1.034676  0.008084              0.001789
2   1.063991  0.014961              0.002290
3   1.017689  0.018821              0.003731
4   1.132133  0.022690              0.003423


Mean performance metric and standard error:
Max error: 1.0529 +- 0.0483
RMSE: 0.0183 +- 0.0072
Exceed boundary rate: 0.0029 +- 0.0008


Model for motor 6:


   Max error      RMSE  Exceed boundary rate
0   1.035807  0.027092              0.002548
1   1.035554  0.009157              0.001789
2   1.060907  0.015565              0.001947
3   1.059725  0.020365              0.006219
4   1.173218  0.023674              0.003423


Mean performance metric and standard error:
Max error: 1.0730 +- 0.0573
RMSE: 0.0192 +- 0.0070
Exceed boundary rate: 0.0032 +- 0.0018


Model for motor 6:


   Max error      RMSE  Exceed boundary rate
0   1.152473  0.027775              0.004994
1   1.177312  0.011532              0.002683
2   1.066143  0.015980              0.002634
3   1.080876  0.021900              0.008706
4   1.184580  0.024442              0.003423


Mean performance metric and standard error:
Max error: 1.1323 +- 0.0552
RMSE: 0.0203 +- 0.0065
Exceed boundary rate: 0.0045 +- 0.0025



=== Summary ===

window_size = 1:
  Max error:           11.6147 +- 2.9111
  RMSE:                28.8224 +- 8.8472
  Exceed boundary rate: 0.8325 +- 0.1312

window_size = 5:
  Max error:           1.0529 +- 0.0483
  RMSE:                0.0183 +- 0.0072
  Exceed boundary rate: 0.0029 +- 0.0008

window_size = 10:
  Max error:           1.0730 +- 0.0573
  RMSE:                0.0192 +- 0.0070
  Exceed boundary rate: 0.0032 +- 0.0018

window_size = 20:
  Max error:           1.1323 +- 0.0552
  RMSE:                0.0203 +- 0.0065
  Exceed boundary rate: 0.0045 +- 0.0025


With window_size = 1 the performance is poor: the RMSE is around 28 and almost every point exceeds the boundary. Increasing it to 5 improves the results considerably, with the RMSE dropping to about 0.018 and the exceed rate to around 0.3%. For window_size = 10 and 20 the results are very similar, with a small increase in the error. The best value for this motor is therefore around window_size = 5.

A larger window helps because the model can use the previous measurements and not only the current instant. The temperature changes slowly and depends on the past behavior of the motor, so providing the last few time steps lets the model follow the trend instead of estimating it from a single point.

Increasing the window indefinitely does not keep improving the performance. Each additional step adds more features, since the variables of all the motors are concatenated, and a large number of features makes the model prone to overfitting. This explains why the error rises slightly after window_size = 5.